# CopositivityDiscriminants.jl – Getting started

This notebook shows how to use `CopositivityDiscriminants.jl` for:

- checking copositivity of polynomials,
- understanding the `CoposCheckResult` output,
- testing nonseparable signed supports.

Run the cells in order.

In [3]:
using Pkg;

# Uncomment this the first time to install from GitHub
#Pkg.add(url="https://github.com/joan-ferrer/CopositivityDiscriminants.jl.git")

using CopositivityDiscriminants;
using HomotopyContinuation;

const HC = HomotopyContinuation #For abbreviation

HomotopyContinuation

## Basic example: checking copositivity of a polynomial

We start with a simple homogeneous polynomial in two variables. We use
`HomotopyContinuation.@polyvar` to create the variables and build an
`Expression`, then call `check_copositivity`.

In [4]:
# Define variables and a test polynomial
HC.@var x y

f =3 + x^4 + y^4 + x^4*y^4 - 2x^2*y - 0.5x*y #The polynomial needs to have full dimensional support
                                             # and no negative terms on the boundaryof the Newton polytope   

f

3 - 0.5*x*y - 2*x^2*y + x^4*y^4 + x^4 + y^4

In [5]:
# Run the copositivity check
res = check_copositivity(f)

res


Certifying 36 solutions... 100%|████████████████████████| Time: 0:00:00
          # processed: 36
   # certified (real): 36 (6)
    # distinct (real): 36 (6)


CoposCheckResult(true, [2.2553726735228 +/- 1.24e-14] + [+/- 1.57e-22]im, :general, System of length 3
 3 variables: t, x, y

 3.0 + 1.0*x^4*y^4 - 0.5*t*x*y - 2.0*t*x^2*y + 1.0*x^4 + 1.0*y^4
 x*(-0.5*t*y + 4.0*x^3*y^4 - 4.0*t*x*y + 4.0*x^3)
 y*(-0.5*t*x - 2.0*t*x^2 + 4.0*x^4*y^3 + 4.0*y^3), CertificationResult
• 36 solution candidates given
• 36 certified solution intervals (6 real, 30 complex)
• 36 distinct certified solution intervals (6 real, 30 complex), HomotopyContinuation.AbstractSolutionCertificate[SolutionCertificate:
solution_candidate = [
  2.2553726735227935 + 2.802596928649634e-45im,
  1.1760107358597354 - 1.0509738482436128e-45im,
  0.8659163158969757 + 1.7516230804060213e-46im,
]
is_certified = true
certified_solution_interval = [
  [2.2553726735228 +/- 1.24e-14] + [+/- 1.57e-22]im,
  [1.17601073585974 +/- 7.75e-15] + [+/- 1.06e-22]im,
  [0.86591631589698 +/- 6.30e-15] + [+/- 6.33e-23]im,
]
precision = 53
is_real = true
index = 23])

The result `res` is a `CoposCheckResult` struct. Let us inspect its fields.


In [6]:
res.copositive #retruns true is the polynomials is certified to be copositive,
               # negative if not, and missing if certification was not possible (i.e 1 lies in the certified interval)

true

In [7]:
res.t_min_interval # the interval where t_min is numerically certified to lie

[2.2553726735228 +/- 1.24e-14] + [+/- 1.57e-22]im

In [8]:
res.method #either general (default) or nonseparable, deppending on which method was used

:general

In [9]:
res.positive_certs # positive certificates found (if any)

1-element Vector{HomotopyContinuation.AbstractSolutionCertificate}:
 SolutionCertificate:
solution_candidate = [
  2.2553726735227935 + 2.802596928649634e-45im,
  1.1760107358597354 - 1.0509738482436128e-45im,
  0.8659163158969757 + 1.7516230804060213e-46im,
]
is_certified = true
certified_solution_interval = [
  [2.2553726735228 +/- 1.24e-14] + [+/- 1.57e-22]im,
  [1.17601073585974 +/- 7.75e-15] + [+/- 1.06e-22]im,
  [0.86591631589698 +/- 6.30e-15] + [+/- 6.33e-23]im,
]
precision = 53
is_real = true
index = 23

## Nonseparable signed supports

The function `nonseparable_support` analyses the *signed support* of a
polynomial. Roughly speaking, we split the exponents of `f` into those with
positive and negative coefficients, take convex hulls, and test a geometric
nonseparability condition using Oscar.


In [10]:
nonsep = nonseparable_support(f; tol=1e-9, verbose=true) #tol is numerical tolerance for detecting when points are in convex hulls

nonsep

true

For polynomials with nonseparable signed support, `check_copositivity` supports a more efficient method.

In [11]:
res_nonsep=check_copositivity(f,nonseparable=true) #using the nonseparable support method by setting nonseparable=true

res_nonsep

CoposCheckResult(true, [2.2553726735228 +/- 1.24e-14] + [+/- 1.57e-22]im, :nonseparable, System of length 3
 3 variables: t, x, y

 3.0 + 1.0*x^4*y^4 - 0.5*t*x*y - 2.0*t*x^2*y + 1.0*x^4 + 1.0*y^4
 x*(-0.5*t*y + 4.0*x^3*y^4 - 4.0*t*x*y + 4.0*x^3)
 y*(-0.5*t*x - 2.0*t*x^2 + 4.0*x^4*y^3 + 4.0*y^3), CertificationResult
• 1 solution candidates given
• 1 certified solution intervals (1 real, 0 complex)
• 1 distinct certified solution intervals (1 real, 0 complex), HomotopyContinuation.AbstractSolutionCertificate[SolutionCertificate:
solution_candidate = [
  2.255372673522794 + 0.0im,
  1.1760107358597356 + 0.0im,
  0.8659163158969757 + 0.0im,
]
is_certified = true
certified_solution_interval = [
  [2.2553726735228 +/- 1.24e-14] + [+/- 1.57e-22]im,
  [1.17601073585974 +/- 7.75e-15] + [+/- 1.06e-22]im,
  [0.86591631589698 +/- 6.30e-15] + [+/- 6.33e-23]im,
]
precision = 53
is_real = true
index = 1])

## Choosing the height function

`check_copositivity` also allows the user to specify a height function. The default sets all the t exponents to 1. 

In [12]:
res_nonsep_h=check_copositivity(f,nonseparable=true,h=[2,5]) #using the nonseparable support method by setting nonseparable=true

res_nonsep_h.system

┌ Info: Applied height function to negative terms:
│   t^2 * (-2.0 * x^2*y)
└   t^5 * (-0.5 * x*y)


System of length 3
 3 variables: t, x, y

 3.0 + 1.0*x^4*y^4 - 2.0*t^2*x^2*y - 0.5*t^5*x*y + 1.0*x^4 + 1.0*y^4
 x*(-0.5*t^5*y + 4.0*x^3*y^4 - 4.0*t^2*x*y + 4.0*x^3)
 y*(-2.0*t^2*x^2 - 0.5*t^5*x + 4.0*x^4*y^3 + 4.0*y^3)